# استخراج بيانات التضاريس الحقيقية
هذا الدفتر يطلع **الارتفاع والانحدار والاتجاه والغطاء الأرضي** لنقاط الانهيارات والنقاط العشوائية، **من نفس المصدر**:
- الارتفاع: Copernicus DEM (دقة 30 متر)
- الغطاء الأرضي: ESA WorldCover 2021 (دقة 10 متر)

## قبل التشغيل
ارفع الملفين من أيقونة المجلد 📁 على اليسار:
1. `landslide_points_clean.csv`
2. `Terrain.csv`

## التشغيل
من القائمة فوق: **Runtime ← Run all** (تشغيل الكل). ياخذ تقريباً 5 إلى 15 دقيقة.

## 1. تثبيت المكتبة

In [ ]:
!pip install -q rasterio

## 2. الدوال

In [ ]:
import math
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

DEM_URL = ("https://copernicus-dem-30m.s3.amazonaws.com/"
           "Copernicus_DSM_COG_10_{ns}{lat:02d}_00_{ew}{lon:03d}_00_DEM/"
           "Copernicus_DSM_COG_10_{ns}{lat:02d}_00_{ew}{lon:03d}_00_DEM.tif")

LC_URL = ("https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/"
          "ESA_WorldCover_10m_2021_v200_{ns}{lat:02d}{ew}{lon:03d}_Map.tif")

LC_NAMES = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare / sparse vegetation", 70: "Snow and ice",
    80: "Permanent water bodies", 90: "Herbaceous wetland",
    95: "Mangroves", 100: "Moss and lichen",
}


def _dem_url(lat, lon):
    la, lo = math.floor(lat), math.floor(lon)
    return DEM_URL.format(ns="N" if la >= 0 else "S", lat=abs(la),
                          ew="E" if lo >= 0 else "W", lon=abs(lo))


def _lc_url(lat, lon):
    la, lo = math.floor(lat / 3) * 3, math.floor(lon / 3) * 3
    return LC_URL.format(ns="N" if la >= 0 else "S", lat=abs(la),
                         ew="E" if lo >= 0 else "W", lon=abs(lo))


def slope_aspect(win, dx, dy):
    """win: مصفوفة 3x3 من الارتفاعات (الصف الأول = الشمال).
    dx, dy: حجم الخلية بالمتر. يرجع الانحدار والاتجاه بالدرجات."""
    a, b, c = win[0]
    d, e, f = win[1]
    g, h, i = win[2]
    dz_east = ((c + 2 * f + i) - (a + 2 * d + g)) / (8 * dx)
    dz_north = ((a + 2 * b + c) - (g + 2 * h + i)) / (8 * dy)
    slope = math.degrees(math.atan(math.hypot(dz_east, dz_north)))
    if dz_east == 0 and dz_north == 0:
        aspect = -1.0                      # أرض مستوية
    else:                                  # اتجاه نزول المنحدر: 0 شمال، 90 شرق
        aspect = (math.degrees(math.atan2(-dz_east, -dz_north)) + 360) % 360
    return round(slope, 2), round(aspect, 1)


_cache = {}
def _open(url):
    if url not in _cache:
        _cache[url] = rasterio.open(url)
    return _cache[url]


def extract_terrain(points):
    """points: جدول فيه عمودين lat و lon.
    يرجع نفس الجدول مع أعمدة التضاريس."""
    rows = []
    for _, p in points.iterrows():
        lat, lon = float(p["lat"]), float(p["lon"])
        out = dict(p)
        try:
            dem = _open(_dem_url(lat, lon))
            r, c = dem.index(lon, lat)
            win = dem.read(1, window=Window(c - 1, r - 1, 3, 3)).astype(float)
            dx = abs(dem.res[0]) * 111320 * math.cos(math.radians(lat))
            dy = abs(dem.res[1]) * 110574
            out["elevation"] = int(round(win[1, 1]))
            out["slope"], out["aspect"] = slope_aspect(win, dx, dy)
        except Exception as err:
            print(f"تعذر حساب الارتفاع للنقطة ({lat}, {lon}): {err}")
            out["elevation"] = out["slope"] = out["aspect"] = np.nan
        try:
            lc = _open(_lc_url(lat, lon))
            r, c = lc.index(lon, lat)
            code = int(lc.read(1, window=Window(c, r, 1, 1))[0, 0])
            out["landcover"], out["landcover_name"] = code, LC_NAMES.get(code, "Unknown")
        except Exception as err:
            print(f"تعذر حساب الغطاء الأرضي للنقطة ({lat}, {lon}): {err}")
            out["landcover"], out["landcover_name"] = np.nan, np.nan
        rows.append(out)
    return pd.DataFrame(rows)


def random_points(lat_min, lat_max, lon_min, lon_max, n, seed=42):
    """نقاط عشوائية داخل مستطيل، مثل نقاط ملف بشاير."""
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        "lat": rng.uniform(lat_min, lat_max, n).round(5),
        "lon": rng.uniform(lon_min, lon_max, n).round(5),
    })

## 3. تجربة على نقطة وحدة (عقبة ضلع)
المفروض يطلع ارتفاع وانحدار منطقي لجرف جبلي.

In [ ]:
extract_terrain(pd.DataFrame({"lat": [18.062696], "lon": [42.493754]}))

## 4. نقاط الانهيارات

In [ ]:
slides = pd.read_csv("landslide_points_clean.csv", encoding="utf-8-sig")

# نقاط NASA مواقعها تقريبية بـ 25-50 كيلو، فما نطلع لها تضاريس
slides = slides[~slides["point_id"].str.startswith("GLC")]

# النقاط اللي بنفس المكان نخليها مرة وحدة (المكان نفسه تضاريسه وحدة)
slides = slides.drop_duplicates(subset=["lat", "lon"])
print("عدد مواقع الانهيارات:", len(slides))

slides_t = extract_terrain(slides[["point_id", "lat", "lon", "location_precision"]])
slides_t["landslide"] = 1
slides_t

## 5. النقاط العشوائية (بدون انهيار)
نحذف أي نقطة عشوائية أقرب من 1 كيلو لانهيار، عشان ما تكون نقطة انهيار بالغلط.

In [ ]:
import numpy as np

random_pts = pd.read_csv("Terrain.csv", encoding="utf-8-sig")[["lat", "lon"]]

def dist_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

near = np.zeros(len(random_pts), dtype=bool)
for _, s in slides.iterrows():
    near |= dist_km(random_pts.lat.values, random_pts.lon.values, s.lat, s.lon) < 1
random_pts = random_pts[~near].reset_index(drop=True)
random_pts.insert(0, "point_id", [f"RND_{i+1}" for i in range(len(random_pts))])
random_pts["location_precision"] = "exact"
print("عدد النقاط العشوائية:", len(random_pts))

random_t = extract_terrain(random_pts)
random_t["landslide"] = 0

## 6. دمج الجدولين وحفظه

In [ ]:
train = pd.concat([slides_t, random_t], ignore_index=True)
train = train.dropna(subset=["elevation", "slope", "aspect", "landcover"])

print("الحجم:", train.shape)
print(train["landslide"].value_counts())
print(train.groupby("landslide")[["elevation", "slope"]].mean().round(1))

train.to_csv("training_table.csv", index=False, encoding="utf-8-sig")

## 7. تحميل الملف

In [ ]:
from google.colab import files
files.download("training_table.csv")